Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import numpy as np
import pandas as pd
from datetime import datetime

RUN_DATE = datetime.now()

DATASET_NAME = "customer_transactions"

NUMERIC_COLUMNS = [
    "quantity",
    "unit_price",
    "discount",
    "tax",
    "shipping_cost",
    "total_sales",
    "profit",
    "profit_margin",
    "inventory_level",
    "website_visits",
    "cart_size",
    "session_duration"
]

PSI_COLUMNS = [
    "total_sales",
    "profit",
    "quantity",
    "customer_age"
]

CATEGORY_COLUMNS = [
    "customer_segment",
    "loyalty_tier",
    "country",
    "region",
    "product_category",
    "payment_method",
    "sales_channel"
]

CORRELATION_COLUMNS = [
    "quantity",
    "unit_price",
    "discount",
    "total_sales",
    "profit"
]

Load Silver

In [0]:
df = spark.table("customer_transactions_silver")

display(df.limit(5))

customer_id,customer_name,customer_age,customer_gender,customer_segment,loyalty_tier,country,state,city,region,product_id,product_name,product_category,subcategory,brand,transaction_id,order_date,shipment_date,payment_method,sales_channel,currency,quantity,unit_price,discount,tax,shipping_cost,total_sales,profit,profit_margin,warehouse,delivery_days,supplier,inventory_level,return_flag,website_visits,cart_size,coupon_used,session_duration,load_date
CUST100125,David Robinson,47.0,Male,Small Business,Gold,Germany,Hesse,Frankfurt,Europe,PRD20740,Zentek Cameras 938,Electronics,Cameras,Zentek,TXN21000000,2026-02-13T16:28:00Z,2026-02-19T16:28:00Z,Credit Card,In-Store,EUR,3,1163.31,0.134,275.79,11.95,3310.02,554.2,0.1674,WH-APAC-02,6.0,Anchor Wholesale,344.0,0,3,5,0.0,6.8,2026-01-02
CUST114825,Karen Jackson,50.0,Female,Small Business,Silver,Canada,Quebec,Quebec City,North America,PRD20313,DriveMax Accessories 943,Automotive,Accessories,DriveMax,TXN21000001,2026-02-10T15:56:00Z,2026-02-14T15:56:00Z,Gift Card,In-Store,CAD,3,257.66,0.08,107.5,13.34,831.98,244.88,0.2943,WH-APAC-01,4.0,Pinnacle Supply Co,514.0,0,9,6,0.0,10.6,2026-01-02
CUST115961,Wei Rivera,53.0,Female,Consumer,Bronze,Germany,Hesse,Frankfurt,Europe,PRD21074,Domus Appliances 919,Home & Kitchen,Appliances,Domus,TXN21000002,2026-02-01T00:34:00Z,2026-02-06T00:34:00Z,Digital Wallet,Online,EUR,4,763.76,0.19,284.77,1.61,2760.96,727.1,0.2634,WH-LATAM-01,5.0,Meridian Traders,312.0,0,4,4,1.0,10.1,2026-01-02
CUST114068,Sarah Hill,59.0,Male,Consumer,Platinum,Vietnam,Ho Chi Minh,Ho Chi Minh City,Asia Pacific,PRD20919,KidJoy Board Games 223,Toys & Games,Board Games,KidJoy,TXN21000003,2026-02-15T03:03:00Z,2026-02-21T03:03:00Z,Credit Card,Online,VND,4,45.46,0.038,23.26,10.97,209.16,84.16,0.4024,WH-NA-01,6.0,Meridian Traders,154.0,0,5,7,0.0,7.7,2026-01-02
CUST123541,Linda Rivera,30.0,Female,Home Office,Bronze,India,Delhi,New Delhi,Asia Pacific,PRD20962,Zentek Security Cameras 608,Smart Home,Security Cameras,Zentek,TXN21000004,2026-02-18T21:18:00Z,2026-02-22T21:18:00Z,Bank Transfer,Call Center,INR,4,155.67,0.033,89.38,3.45,694.96,213.58,0.3073,WH-LATAM-01,4.0,Meridian Traders,588.0,0,3,5,0.0,13.4,2026-01-02


Split Daily Data

In [0]:
day1 = df.filter(F.col("load_date") == "2026-01-01")

day2 = df.filter(F.col("load_date") == "2026-01-02")

day3 = df.filter(F.col("load_date") == "2026-01-03")

Volume Drift

In [0]:
def calculate_volume_drift(old_df, new_df):

    old_count = old_df.count()

    new_count = new_df.count()

    change = ((new_count - old_count) / old_count) * 100

    score = min(abs(change), 100)

    return {
        "old": old_count,
        "new": new_count,
        "change": change,
        "score": score
    }


volume = calculate_volume_drift(day1, day2)

volume_score = volume["score"]

print(volume)

{'old': 100000, 'new': 100000, 'change': 0.0, 'score': 0.0}


Numeric Drift

In [0]:
numeric_results = []

changes = []

for column in NUMERIC_COLUMNS:

    old_mean = day1.select(F.avg(column)).first()[0]

    new_mean = day2.select(F.avg(column)).first()[0]

    if old_mean == 0:

        pct_change = 0

    else:

        pct_change = ((new_mean-old_mean)/old_mean)*100

    changes.append(abs(pct_change))

    numeric_results.append({

        "metric":column,

        "old":old_mean,

        "new":new_mean,

        "change":pct_change

    })

numeric_score = min(np.mean(changes),100)

print("Numeric Score =",numeric_score)

display(pd.DataFrame(numeric_results))

Numeric Score = 14.44149274602733


metric,old,new,change
quantity,3.10025,3.60236,16.195790662043386
unit_price,336.79557059999996,343.25798169999985,1.9187933761976514
discount,0.08002032999999993,0.1250470900000001,56.269150602103515
tax,119.76505219999999,135.16274130000002,12.856579458828172
shipping_cost,8.7351803,8.0739049,-7.570254731891443
total_sales,1089.0726423000006,1225.2162981999995,12.500879244609312
profit,331.54173779999996,301.6593229,-9.013168326343965
profit_margin,0.34204205000000004,0.288424915,-15.67559748867136
inventory_level,490.4551,344.06731,-29.847337707366073
website_visits,5.0004,4.99122,-0.1835853131749409


PSI Drift

In [0]:
import numpy as np


def calculate_psi(
    expected,
    actual,
    bins=10
):

    expected = np.array(expected)
    actual = np.array(actual)


    # Remove nulls
    expected = expected[~pd.isnull(expected)]
    actual = actual[~pd.isnull(actual)]


    # Create bins from combined distribution
    breakpoints = np.linspace(
        min(
            expected.min(),
            actual.min()
        ),
        max(
            expected.max(),
            actual.max()
        ),
        bins + 1
    )


    expected_counts = np.histogram(
        expected,
        breakpoints
    )[0]


    actual_counts = np.histogram(
        actual,
        breakpoints
    )[0]


    # Convert to percentages
    expected_pct = (
        expected_counts /
        len(expected)
    )


    actual_pct = (
        actual_counts /
        len(actual)
    )


    # Avoid zero division
    expected_pct = np.where(
        expected_pct == 0,
        0.0001,
        expected_pct
    )


    actual_pct = np.where(
        actual_pct == 0,
        0.0001,
        actual_pct
    )


    psi = np.sum(
        (
            actual_pct - expected_pct
        )
        *
        np.log(
            actual_pct /
            expected_pct
        )
    )


    return psi

In [0]:
psi_results = []

psi_scores = []

for column in PSI_COLUMNS:

    old_values = day1.select(column).toPandas()[column]

    new_values = day2.select(column).toPandas()[column]

    psi = calculate_psi(
        old_values,
        new_values,
        bins=10
    )

    psi_scores.append(
        min(psi*100,100)
    )

    psi_results.append({

        "metric":column,

        "psi":psi

    })

psi_score = np.mean(psi_scores)

display(pd.DataFrame(psi_results))

metric,psi
total_sales,0.007291020475397499
profit,0.011231309703582364
quantity,0.10384442476279979
customer_age,0.18000357004543574


Category Drift

In [0]:
category_results=[]

changes=[]

for column in CATEGORY_COLUMNS:

    old_dist = (
        day1.groupBy(column)
        .count()
        .withColumnRenamed("count","old")
    )

    new_dist = (
        day2.groupBy(column)
        .count()
        .withColumnRenamed("count","new")
    )

    joined = (
        old_dist
        .join(
            new_dist,
            column,
            "outer"
        )
        .fillna(0)
    )

    total_old = joined.select(F.sum("old")).first()[0]

    total_new = joined.select(F.sum("new")).first()[0]

    joined = joined.withColumn(
        "old_pct",
        F.col("old")/total_old
    )

    joined = joined.withColumn(
        "new_pct",
        F.col("new")/total_new
    )

    joined = joined.withColumn(
        "change",
        F.abs(F.col("new_pct")-F.col("old_pct"))
    )

    avg_change = joined.select(
        F.avg("change")
    ).first()[0]

    changes.append(avg_change*100)

    category_results.append({

        "column":column,

        "change":avg_change*100

    })

category_score=min(np.mean(changes),100)

display(pd.DataFrame(category_results))

column,change
customer_segment,0.48199999999999943
loyalty_tier,0.34699999999999975
country,0.6492727272727276
region,1.239
product_category,2.619333333333333
payment_method,4.750666666666667
sales_channel,0.09919999999999986


Correlation Drift

In [0]:
corr1 = (
    day1
    .select(CORRELATION_COLUMNS)
    .toPandas()
    .corr()
)

corr2 = (
    day2
    .select(CORRELATION_COLUMNS)
    .toPandas()
    .corr()
)

corr_diff = abs(corr2-corr1)

correlation_score = min(
    corr_diff.values.mean()*100,
    100
)

display(corr_diff)

quantity,unit_price,discount,total_sales,profit
0.0,7.941667975869743E-4,0.0026081590703301277,0.020438293253940754,0.024179074856408633
7.941667975869743E-4,0.0,0.0035230247244438665,0.010631590706233096,0.0732892392556932
0.0026081590703301277,0.0035230247244438665,0.0,0.017734308164774656,0.1121013132741975
0.020438293253940754,0.010631590706233096,0.017734308164774656,0.0,0.06444953414725929
0.024179074856408633,0.0732892392556932,0.1121013132741975,0.06444953414725929,0.0


KPI Drift

In [0]:
def kpis(df):

    return {

        "Revenue":
        df.select(F.sum("total_sales")).first()[0],

        "Profit":
        df.select(F.sum("profit")).first()[0],

        "AOV":
        df.select(F.avg("total_sales")).first()[0],

        "ReturnRate":
        df.select(F.avg("return_flag")).first()[0],

        "Delivery":
        df.select(F.avg("delivery_days")).first()[0]

    }

old=kpis(day1)

new=kpis(day2)

kpi_results=[]

changes=[]

for metric in old.keys():

    if old[metric]==0:

        pct=0

    else:

        pct=((new[metric]-old[metric])/old[metric])*100

    changes.append(abs(pct))

    kpi_results.append({

        "metric":metric,

        "old":old[metric],

        "new":new[metric],

        "change":pct

    })

kpi_score=min(np.mean(changes),100)

display(pd.DataFrame(kpi_results))

metric,old,new,change
Revenue,1.089072642300001E8,1.2252162982000053E8,12.5008792446098
Profit,3.3154173779999968E7,3.016593228999975E7,-9.013168326344644
AOV,1089.0726423000006,1225.2162981999995,12.500879244609312
ReturnRate,0.05224,0.06959,33.21209800918836
Delivery,4.99275,5.80661,16.300836212508138


Overall Drift Score

In [0]:
overall_score=(

volume_score*0.10+

numeric_score*0.25+

psi_score*0.25+

category_score*0.15+

correlation_score*0.10+

kpi_score*0.15

)

def severity(score):

    if score<30:

        return "LOW"

    elif score<70:

        return "MEDIUM"

    return "HIGH"

severity_level=severity(overall_score)

print("Overall Score:",overall_score)

print("Severity:",severity_level)

Overall Score: 8.48810407063699
Severity: LOW


Delta <- Results

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("run_date", TimestampType(), True),
    StructField("comparison_period", StringType(), True),
    StructField("metric_name", StringType(), True),
    StructField("drift_type", StringType(), True),
    StructField("old_value", DoubleType(), True),
    StructField("new_value", DoubleType(), True),
    StructField("change_percentage", DoubleType(), True),
    StructField("drift_score", DoubleType(), True),
    StructField("severity", StringType(), True),
    StructField("description", StringType(), True)
])

In [0]:
results = [

(
RUN_DATE,
"day1_vs_day2",
"Volume",
"Volume Drift",
float(volume["old"]),
float(volume["new"]),
float(volume["change"]),
float(volume_score),
severity(volume_score),
"Daily record count change"
),

(
RUN_DATE,
"day1_vs_day2",
"Numeric Features",
"Numeric Drift",
None,
None,
None,
float(numeric_score),
severity(numeric_score),
"Statistical change across numerical features"
),

(
RUN_DATE,
"day1_vs_day2",
"Population Stability",
"PSI",
None,
None,
None,
float(psi_score),
severity(psi_score),
"Population distribution change"
),

(
RUN_DATE,
"day1_vs_day2",
"Categorical Features",
"Category Drift",
None,
None,
None,
float(category_score),
severity(category_score),
"Category distribution change"
),

(
RUN_DATE,
"day1_vs_day2",
"Correlation",
"Correlation Drift",
None,
None,
None,
float(correlation_score),
severity(correlation_score),
"Correlation relationship change"
),

(
RUN_DATE,
"day1_vs_day2",
"Business KPI",
"KPI Drift",
None,
None,
None,
float(kpi_score),
severity(kpi_score),
"Revenue, profit and operational KPI change"
),

(
RUN_DATE,
"day1_vs_day2",
"Overall",
"Composite",
None,
None,
None,
float(overall_score),
severity(overall_score),
"Weighted drift score"
)

]

In [0]:
drift_df = spark.createDataFrame(
    results,
    schema=schema
)

display(drift_df)

run_date,comparison_period,metric_name,drift_type,old_value,new_value,change_percentage,drift_score,severity,description
2026-07-29T09:10:35.099807Z,day1_vs_day2,Volume,Volume Drift,100000.0,100000.0,0.0,0.0,LOW,Daily record count change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Numeric Features,Numeric Drift,null,null,null,14.44149274602733,LOW,Statistical change across numerical features
2026-07-29T09:10:35.099807Z,day1_vs_day2,Population Stability,PSI,null,null,null,7.559258124680385,LOW,Population distribution change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Categorical Features,Category Drift,null,null,null,1.4552103896103894,LOW,Category distribution change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Correlation,Correlation Drift,null,null,null,2.637989634006945,LOW,Correlation relationship change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Business KPI,KPI Drift,null,null,null,16.705572207452047,LOW,"Revenue, profit and operational KPI change"
2026-07-29T09:10:35.099807Z,day1_vs_day2,Overall,Composite,null,null,null,8.48810407063699,LOW,Weighted drift score


In [0]:
(
    drift_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("metric_drift_results_delta")
)

In [0]:
(
    drift_df.write
    .format("delta")
    .mode("append")
    .save("abfss://metadata@<your_storage_account>.dfs.core.windows.net/metric_drift_results")
)

In [0]:
%sql
SELECT
    metric_name,
    drift_type,
    drift_score,
    severity
FROM metric_drift_results_delta
ORDER BY run_date DESC;

metric_name,drift_type,drift_score,severity
Correlation,Correlation Drift,2.637989634006945,LOW
Categorical Features,Category Drift,1.4552103896103894,LOW
Overall,Composite,8.48810407063699,LOW
Volume,Volume Drift,0.0,LOW
Business KPI,KPI Drift,16.705572207452047,LOW
Population Stability,PSI,7.559258124680385,LOW
Numeric Features,Numeric Drift,14.44149274602733,LOW
Correlation,Correlation Drift,2.63798963400715,LOW
Numeric Features,Numeric Drift,14.441492746027329,LOW
Overall,Composite,8.476506192899386,LOW


In [0]:
display(drift_df)

drift_df.printSchema()

run_date,comparison_period,metric_name,drift_type,old_value,new_value,change_percentage,drift_score,severity,description
2026-07-29T09:10:35.099807Z,day1_vs_day2,Volume,Volume Drift,100000.0,100000.0,0.0,0.0,LOW,Daily record count change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Numeric Features,Numeric Drift,null,null,null,14.44149274602733,LOW,Statistical change across numerical features
2026-07-29T09:10:35.099807Z,day1_vs_day2,Population Stability,PSI,null,null,null,7.559258124680385,LOW,Population distribution change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Categorical Features,Category Drift,null,null,null,1.4552103896103894,LOW,Category distribution change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Correlation,Correlation Drift,null,null,null,2.637989634006945,LOW,Correlation relationship change
2026-07-29T09:10:35.099807Z,day1_vs_day2,Business KPI,KPI Drift,null,null,null,16.705572207452047,LOW,"Revenue, profit and operational KPI change"
2026-07-29T09:10:35.099807Z,day1_vs_day2,Overall,Composite,null,null,null,8.48810407063699,LOW,Weighted drift score


root
 |-- run_date: timestamp (nullable = true)
 |-- comparison_period: string (nullable = true)
 |-- metric_name: string (nullable = true)
 |-- drift_type: string (nullable = true)
 |-- old_value: double (nullable = true)
 |-- new_value: double (nullable = true)
 |-- change_percentage: double (nullable = true)
 |-- drift_score: double (nullable = true)
 |-- severity: string (nullable = true)
 |-- description: string (nullable = true)



In [0]:
(
    drift_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "mdo_dev_dbx.default.metric_drift_results_delta"
    )
)